# ⚡ Week 3 — Groq Integration
> **Goal:** Use Groq's ultra-fast inference as the LLM backend and learn to stream responses.

---
### What you will learn
1. Groq API setup
2. Groq via raw SDK (groq Python package)
3. Streaming responses
4. Groq via PydanticAI
5. Latency benchmark vs a slower API
6. Mini-project: fast chatbot with streaming

## 1 — Setup

In [36]:
%pip install groq pydantic-ai python-dotenv rich --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\Nidhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [37]:
%pip install groq pydantic-ai python-dotenv rich

import os
import time
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\Nidhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## 2 — Groq raw SDK

In [38]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What makes Groq fast?"},
    ],
    max_tokens=256,
)

print(response.choices[0].message.content)
print(f"\nTokens used: {response.usage.total_tokens}")

Groq is a leading technology company focused on developing high-performance AI hardware and software solutions, primarily the Groq Accelerator, which is a purpose-built AI processor designed for large-scale deployments in data centers.

Groq's processor is fast due to several key innovations:

1. **High-Performance, High-Utilization Architecture**: The Groq Accelerator features a custom-designed processor architecture that is optimized for AI workloads, allowing for high utilization of the hardware resources.

2. **High Clock Rate**: The Groq Accelerator operates at extremely high clock rates (up to 1.4 GHz), enabling high-performance processing of complex AI computations.

3. **Multi-Chip and Multi-Die Packaging**: Groq's processor is designed with multiple dies and chips, which enables increased processing capacity and bandwidth between different components.

4. **Low Latency**: The Groq Accelerator is designed to reduce latency, allowing it to handle real-time AI workloads with mini

## 3 — Streaming responses

In [39]:
# assume client is already initialized
import sys

print("Streaming response:\n")

stream = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": "Explain what a neural network is in 5 sentences."}
    ],
    stream=True,
)

for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="", flush=True)

Streaming response:

A neural network is a complex system of interconnected nodes or "neurons" that work together to process and analyze data. Inspired by the human brain's neural structure, these networks are designed to recognize patterns and make decisions by learning from large datasets. Each node in the network receives one or more inputs, performs a calculation on those inputs, and then sends the output to other nodes, allowing the network to learn and improve its performance over time. Neural networks can be trained on a wide range of tasks, including image recognition, speech recognition, and decision-making, and have numerous applications in fields like artificial intelligence, computer vision, and robotics. Through a process called backpropagation, the network adjusts the connections between nodes to refine its performance, enabling it to learn and adapt to new data and situations.

## 4 — Groq via PydanticAI (the clean way)

In [40]:
pip install pydantic-ai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\Nidhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [41]:
from pydantic import BaseModel
from pydantic_ai import Agent

class CodeReview(BaseModel):
    issues: list[str]
    suggestions: list[str]
    quality_score: int

agent = Agent(
    model="groq:llama-3.3-70b-versatile",
    output_type=CodeReview,
    system_prompt="""
Return ONLY valid JSON.

No tags, no function wrappers, no markdown.
Strict JSON only.
""",
    retries=3,
)

code = """
def get_user(id):
    import sqlite3
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users WHERE id = ?", (id,))
    return cursor.fetchone()
"""

review = (await agent.run(f"Review this code:\n{code}")).output

print(review.quality_score)
print(review.issues)
print(review.suggestions)

6
["The function opens a new database connection every time it's called, which can be inefficient if the function is called frequently. It would be more efficient to establish a connection once and reuse it.", "The function does not close the database connection, which can lead to resource leaks. It's a good practice to close the connection when you're done with it.", 'The function does not handle potential errors that may occur during database operations.', 'The function returns the entire row from the database, which may contain more data than needed. It would be more efficient to only retrieve the necessary columns.']
['Establish a database connection once and reuse it.', "Close the database connection when you're done with it.", 'Handle potential errors during database operations.', 'Only retrieve the necessary columns from the database.']


## 6 — Latency benchmark

In [42]:
import time
from groq import Groq
import os

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

models = ["llama-3.1-8b-instant", "llama-3.3-70b-versatile"]

prompt = "Explain machine learning in one line."

for model in models:
    start = time.perf_counter()

    client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )

    print(model, time.perf_counter() - start)

llama-3.1-8b-instant 0.3467614000001049
llama-3.3-70b-versatile 0.6099616999999853


## 7 — Async streaming with PydanticAI

In [43]:
from pydantic_ai import Agent

stream_agent = Agent(
    model="groq:llama-3.1-8b-instant",
    system_prompt="You are a helpful assistant.",
)

async def stream_response(question: str):
    print(f"Q: {question}\nA: ", end="", flush=True)

    async with stream_agent.run_stream(question) as response:
        async for chunk in response.stream_text(delta=True):
            print(chunk, end="", flush=True)

    print()

await stream_response("What are 3 benefits of type hints in Python?")

Q: What are 3 benefits of type hints in Python?
A: Type hints in Python can greatly improve code readability and maintainability. Here are 3 benefits of using type hints:

1. **Better Code Understandability**: Type hints allow other developers (and even your future self) to quickly understand the expected input parameter types and return types of a function without having to read the entire code. This makes the code more self-explanatory and easier to understand.

2. **Better Auto Completion and Code Analysis Tools**: Modern IDEs and code analysis tools such as Pylint, mypy, and type checkers like Pyright can use type hints to provide more accurate auto-completion suggestions and catch type-related errors.

3. **More Robust Code**: Using type hints helps catch type-related errors at runtime, as seen in Python 3.5 and above, using Optional type and runtime type checking with tools like typeguard. This ensures that your code is more robust and less prone to crashing due to type-related i

## 🏆 Mini-Project: Fast Streaming Chatbot

In [44]:
from pydantic_ai import Agent

class StreamingChatbot:
    def __init__(self, model="groq:llama-3.1-8b-instant"):
        self.agent = Agent(
            model=model,
            system_prompt="Answer in 2-4 sentences, be concise."
        )
        self.history = []

    async def chat(self, msg):
        print(f"You: {msg}\nBot: ", end="", flush=True)

        async with self.agent.run_stream(msg, message_history=self.history) as r:
            out = ""
            async for c in r.stream_text(delta=True):
                print(c, end="", flush=True)
                out += c

        print()
        self.history = r.new_messages()
        return out


bot = StreamingChatbot()

await bot.chat("Hi, I am learning AI agents.")
await bot.chat("What did I just tell you?")
await bot.chat("Give me one tip for learning AI faster.")

You: Hi, I am learning AI agents.
Bot: AI agents are a fundamental concept in artificial intelligence. They are software programs that perceive their environment, take actions, and make decisions based on a set of rules or goals. You can explore various types of AI agents, such as reactive, model-based, and learning agents, each with its strengths and applications.
You: What did I just tell you?
Bot: You just told me that you are learning about AI agents.
You: Give me one tip for learning AI faster.
Bot: One tip for learning AI faster is to focus on understanding the underlying concepts and principles rather than just memorizing formulas or algorithms. This will help you to build a solid foundation and make it easier to learn and apply more advanced concepts as you go.


'One tip for learning AI faster is to focus on understanding the underlying concepts and principles rather than just memorizing formulas or algorithms. This will help you to build a solid foundation and make it easier to learn and apply more advanced concepts as you go.'

## Week 3 — Summary
| Topic | Key Takeaway |
|---|---|
| Groq SDK | `Groq(api_key=...)` → same interface as OpenAI |
| PydanticAI + Groq | Just use `"groq:model-name"` |
| Streaming | `stream=True` (SDK) or `agent.run_stream()` (PydanticAI) |
| Speed | 8B models ~10x faster than typical OpenAI endpoints |
| Model choice | 8B = speed, 70B = quality, Mixtral = long context |

**Next: Week 4 — OpenRouter (control multiple models from one API)**